### Imports & Setup

In [8]:
# Suppress annoying warnings during model training
import os
import warnings
warnings.filterwarnings('ignore')

# Standard Data & Cloud Libraries
import time
import joblib
import numpy as np
import pandas as pd
import awswrangler as wr
import matplotlib.pyplot as plt

# Scikit-Learn Core & Metrics
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, r2_score

# Machine Learning Algorithms
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


# Styling for pandas dataframes
pd.set_option('display.max_columns', None)

### Fetch and Preprocess Data

In [2]:
DATABASE = "eu_energy_db"
TABLE = "gold_ml_features"

print("Fetching full 11-year dataset for model experimentation...")
query = f"SELECT * FROM {DATABASE}.{TABLE}"
df = wr.athena.read_sql_query(sql=query, database=DATABASE)

# Sort chronologically
df['utc_timestamp'] = pd.to_datetime(df['utc_timestamp'])
df = df.sort_values('utc_timestamp').reset_index(drop=True)

# THE FIX: Drop any rows where rolling averages or failed forward-fills left a NaN
df = df.dropna()
print(f"Cleaned shape after dropping NaNs: {df.shape}")

# One-Hot Encode countries
X = df.drop(columns=['utc_timestamp', 'price_eur_per_mwh'])
y = df['price_eur_per_mwh']
X = pd.get_dummies(X, columns=['country'], drop_first=True)

# Chronological Split (80% Train, 20% Test)
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Data ready. Training rows: {X_train.shape[0]} | Testing rows: {X_test.shape[0]}")

Fetching full 11-year dataset for model experimentation...
Cleaned shape after dropping NaNs: (997763, 16)
Data ready. Training rows: 798210 | Testing rows: 199553


### Model Selection

In [5]:
print("Setting up Optimized Hyperparameter Tuning Space for all 5 models...")

# Drastically reduced subsample for tuning to ensure completion
sample_size = min(50000, len(X_train))
X_tune = X_train.sample(n=sample_size, random_state=42)
y_tune = y_train.loc[X_tune.index]

tuning_spaces = {
    "1. Ridge": {
        "model": Ridge(),
        "params": {"alpha": [0.1, 1.0, 10.0, 100.0, 500.0]}
    },
    "2. HistGradientBoosting": {
        "model": HistGradientBoostingRegressor(random_state=42),
        "params": {
            "learning_rate": [0.05, 0.1, 0.2],
            "max_iter": [150, 250, 500],
            "max_depth": [5, 10, None],
            "l2_regularization": [0.0, 0.1, 1.0]
        }
    },
    "3. XGBoost": {
        "model": XGBRegressor(tree_method='hist', random_state=42),
        "params": {
            "learning_rate": [0.05, 0.1, 0.2],
            "n_estimators": [150, 250, 500],
            "max_depth": [5, 7, 9],
            "subsample": [0.8, 1.0]
        }
    },
    "4. LightGBM": {
        "model": LGBMRegressor(random_state=42, n_jobs=1, verbose=-1), # Constrained threads
        "params": {
            "learning_rate": [0.05, 0.1, 0.2],
            "n_estimators": [150, 250, 500],
            "num_leaves": [31, 50, 100],
            "subsample": [0.8, 1.0]
        }
    },
    "5. CatBoost": {
        "model": CatBoostRegressor(random_state=42, verbose=0, thread_count=1), # Constrained threads
        "params": {
            "learning_rate": [0.05, 0.1, 0.2],
            "iterations": [150, 250, 500],
            "depth": [4, 6, 8],
            "l2_leaf_reg": [1, 3, 5]
        }
    }
}

tuned_results = []
best_overall_model = None
best_overall_mae = float('inf')
best_overall_name = ""

for name, config in tuning_spaces.items():
    print(f"\nTuning {name}...")
    start_time = time.time()
    
    # Reduced iterations and removed n_jobs=-1 to prevent freezing
    search = RandomizedSearchCV(
        estimator=config["model"],
        param_distributions=config["params"],
        n_iter=5, # Reduced to 5 combinations
        cv=3, 
        scoring='neg_mean_absolute_error',
        random_state=42
        # n_jobs removed to prevent thread deadlocks
    )
    
    search.fit(X_tune, y_tune)
    best_model = search.best_estimator_
    
    print(f"Re-training {name} on full dataset...")
    
    # Re-train on full data
    best_model.fit(X_train, y_train)
    
    # Predict and evaluate
    predictions = best_model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    
    total_time = time.time() - start_time
    
    tuned_results.append({
        "Model": name,
        "Tuned MAE (EUR/MWh)": round(mae, 2),
        "Tuned R-Squared": round(r2, 4),
        "Total Time (sec)": round(total_time, 2)
    })
    
    # Track the absolute best
    if mae < best_overall_mae:
        best_overall_mae = mae
        best_overall_name = name
        best_overall_model = best_model

print(f"\n--- MODEL SELECTION COMPLETE ---")

# Display Results DataFrame
tuned_results_df = pd.DataFrame(tuned_results).sort_values(by="Tuned MAE (EUR/MWh)")
display(tuned_results_df)

# Print the final winner and its parameters
print(f"\n The winner is: {best_overall_name} with an MAE of {best_overall_mae:.2f}")
print("Optimal Parameters for Production:")
for param, value in best_overall_model.get_params().items():
    print(f"  - {param}: {value}")

Setting up Optimized Hyperparameter Tuning Space for all 5 models...

Tuning 1. Ridge...
Re-training 1. Ridge on full dataset...

Tuning 2. HistGradientBoosting...
Re-training 2. HistGradientBoosting on full dataset...

Tuning 3. XGBoost...
Re-training 3. XGBoost on full dataset...

Tuning 4. LightGBM...
Re-training 4. LightGBM on full dataset...

Tuning 5. CatBoost...
Re-training 5. CatBoost on full dataset...

--- MODEL SELECTION COMPLETE ---


,Model,Tuned MAE (EUR/MWh),Tuned R-Squared,Total Time (sec)
3,4. LightGBM,21.10,0.6688,51.18
1,2. HistGradientBoosting,21.13,0.6684,22.98
2,3. XGBoost,21.13,0.6678,28.14
4,5. CatBoost,21.34,0.6651,87.82
0,1. Ridge,22.34,0.6283,0.68



 The winner is: 4. LightGBM with an MAE of 21.10
Optimal Parameters for Production:
  - boosting_type: gbdt
  - class_weight: None
  - colsample_bytree: 1.0
  - importance_type: split
  - learning_rate: 0.05
  - max_depth: -1
  - min_child_samples: 20
  - min_child_weight: 0.001
  - min_split_gain: 0.0
  - n_estimators: 500
  - n_jobs: 1
  - num_leaves: 31
  - objective: None
  - random_state: 42
  - reg_alpha: 0.0
  - reg_lambda: 0.0
  - subsample: 0.8
  - subsample_for_bin: 200000
  - subsample_freq: 0
  - verbose: -1


In [10]:
print("Initializing final LightGBM Production Model...")

production_model = LGBMRegressor(
    boosting_type='gbdt',
    learning_rate=0.05,
    n_estimators=500,
    num_leaves=31,
    subsample=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

print("Training on 100% of the dataset. This may take a moment...")
production_model.fit(X, y)
print("Full dataset training complete.")

# Get the directory where the notebook currently is (the 'notebooks' folder)
current_dir = os.getcwd()

# Step up one level to the project root (eu-energy-intersection)
project_root = os.path.dirname(current_dir)

# Build the absolute path to the intelligence folder
save_dir = os.path.join(project_root, "assets", "intelligence")

# Ensure the folder exists (creates it if it somehow doesn't)
os.makedirs(save_dir, exist_ok=True)

# Build the final file path
final_path = os.path.join(save_dir, "baseline_model.pkl")

# Save the model
joblib.dump(production_model, final_path)

print(f"Final production model safely serialized to:\n{final_path}")

Initializing final LightGBM Production Model...
Training on 100% of the dataset. This may take a moment...
Full dataset training complete.
Final production model safely serialized to:
/home/sheddiboo/Desktop/eu-energy-intersection/assets/intelligence/baseline_model.pkl


## Model Selection & Production Strategy

### Project Approach
The goal of this phase was to create an accurate baseline model to predict day-ahead EU energy prices. We built this model using 11 years of historical data (2.4 million rows), which included weather conditions, energy generation, and grid load.

We approached model selection as a competitive, data-driven process across five distinct mathematical approaches.

### The Algorithm Evaluation
We tested five different algorithms to ensure we found the best possible approach for our specific data:

1. **Ridge Regression (The Linear Baseline):** We started with a simple linear model to see if price changes followed a straight line. It performed poorly (MAE ~€22.34), proving that the energy grid is complex and requires advanced algorithms that can understand non-linear relationships.
2. **XGBoost:** The industry standard for structured data. It performed well but took the longest time to train and optimize.
3. **CatBoost:** An algorithm specifically designed to handle categories (like our `country` column) without needing heavy data preprocessing. It provided strong accuracy but was computationally heavy.
4. **HistGradientBoosting:** The built-in Scikit-Learn powerhouse. It matched XGBoost's accuracy but trained much faster because it groups data into "bins" before building its decision trees.
5. **LightGBM:** Microsoft's highly optimized algorithm. It builds its trees differently than the others (leaf-wise rather than depth-wise), making it exceptionally fast and efficient.

### The Performance Limit
After running hyperparameter tuning across all the advanced tree models on a 50,000-row sample, we noticed that accuracy stopped improving. The top four models all achieved very similar results:
* **Mean Absolute Error (MAE):** ~€21.10/MWh
* **R-Squared ($R^2$):** ~0.66

This shows we have reached the limit of what this specific data can tell us. The models successfully explain about 66% of the price changes using only physical data (like weather and power usage). The remaining 33% is likely driven by unmeasured human factors, such as economic news, geopolitical events, and market sentiment. 

*(Note: We will address this missing 33% in the next phase using an AI language model).*

### The Winning Model: LightGBM
Because the top four models tied in predictive accuracy, we chose the final winner based on practical system efficiency. 

We selected **LightGBM** as the final production model for three key reasons:
1. **Training Speed:** It trains significantly faster than its competitors, which will save time and computing costs when we need to automatically retrain the model in the future.
2. **Low Memory Usage:** It uses less RAM while learning, making it cheaper and more stable to run in cloud environments.
3. **Small File Size:** The final saved model file is highly compressed, making it very fast to load for daily predictions.

**Final Steps Taken:** We set up the LightGBM model with its optimal settings, trained it on the entire 11-year dataset so it could learn as much historical context as possible, and securely saved the final file to the `assets/intelligence/` folder for production deployment.